In [9]:
import numpy as np
import pandas as pd
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
from sklearn.model_selection import GridSearchCV

LANDMARK_DAYS = 1200
TARGET_COLS = ["outcome", "event_adverse", "event_transplant", "censored", "days_to_event"]
NUM_COLS = ["BMI_TCR", "INIT_CPRA", "INIT_AGE", "functional_percent",
            "DIALYSIS_DURATION", "AGE_DIALYSIS_INTERACTION"]
NON_ADVERSE_EXITS = ["transplanted", "transplanted_elsewhere", "removed_administrative"]

In [10]:
df = pd.read_csv("../data/processed/task3_encoded_scaled.csv")
df.shape

(494803, 51)

In [11]:
def prep_cox(df):
    d = df[df['days_to_event'] > 0]
    mask = ((d['days_to_event'] >= LANDMARK_DAYS)
            | (d['event_adverse'] == 1)
            | d['outcome'].isin(NON_ADVERSE_EXITS))
    d = d[mask]

    # binary label
    y = ((d['event_adverse'] == 1) & (d['days_to_event'] <= LANDMARK_DAYS)).astype(int)

    # survival outcome
    time = d['days_to_event'].clip(upper=LANDMARK_DAYS).astype(float)
    surv = Surv.from_arrays(event=y.astype(bool).to_numpy(), time=time.to_numpy())

    X = d.drop(columns=TARGET_COLS + ['DIALYSIS_DURATION_LOG']).astype(float)
    X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
        X, y, surv, test_size=0.2, stratify=y, random_state=42
    )
    X_train, X_test = X_train.copy(), X_test.copy()

    scaler = StandardScaler()
    X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
    X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])
    return X_train, X_test, y_train, y_test, s_train, s_test

X_train, X_test, y_train, y_test, s_train, s_test = prep_cox(df)



NameError: name 'train_test_split' is not defined